# Data Engineering Interview Prep: Statistics (Complete 1 - 7)
## Topic: Hypothesis Testing, Probability, Expected Values, and Bayes' Theorem

---

## Table of Contents
1. [Facebook: Coin Fairness Test (Easy)](#statistics-lesson-1-facebook---coin-fairness-test)
2. [Visa: 4 Rolls To 4 (Easy)](#statistics-lesson-2-visa---4-rolls-to-4)
3. [Akuna Capital: Consecutive Fives (Medium)](#statistics-lesson-3-akuna-capital---consecutive-fives)
4. [D.E. Shaw: Biased Coin? (Medium)](#statistics-lesson-4-de-shaw---biased-coin)
5. [Facebook: Product vs. Square (Hard)](#statistics-lesson-5-facebook---product-vs-square)
6. [Google: Minimum of Two Uniform Variables (Hard)](#statistics-lesson-6-google---minimum-of-two-uniform-variables)
7. [Microsoft: Frequentist vs. Bayesian (Hard)](#statistics-lesson-7-microsoft---frequentist-vs-bayesian)

---

### **Statistics Lesson 1: Facebook - "Coin Fairness Test"**

#### **The Problem**
You are given a coin. How would you determine if it is a fair coin?

#### **The Logic (Hypothesis Testing)**
1. **Define the Hypotheses:** H0 (Null): The coin is fair (P(Heads) = 0.5). Ha (Alternative): The coin is biased (P(Heads) != 0.5).
2. **Experiment:** Flip the coin a large number of times (e.g., N = 100).
3. **Calculate:** Use the Binomial Distribution to find the probability of observing your exact result assuming the coin is fair.
4. **Conclude:** If that probability (p-value) is less than your significance threshold (usually alpha = 0.05), you reject the null hypothesis.

#### **Senior Data Engineer Perspective**
* **Data Quality Anomalies:** This exact logic powers Data Quality frameworks like Great Expectations. A Senior DE uses statistical thresholds (like Z-scores based on historical variance) rather than hardcoded rules to prevent false-positive pipeline alerts.

---

### **Statistics Lesson 2: Visa - "4 Rolls To 4"**

#### **The Problem**
What is the probability of rolling at least one 4 in four rolls of a fair six-sided die?

#### **The Logic (Complementary Probability)**
Use the Complement Rule: P(at least one) = 1 - P(none).
1. The probability of not rolling a 4 on a single roll is 5/6.
2. The probability of not rolling a 4 in four consecutive rolls is (5/6)^4.
3. Calculate: (5/6)^4 = 625 / 1296 = ~0.482.
4. Subtract from 1: 1 - 0.482 = 0.518.
**Answer: ~51.8%**

#### **Senior Data Engineer Perspective**
* **SLA and Failure Probabilities:** If a microservice has a 1/6 chance of failing on any given day, what is the probability it ruins your weekly SLA by failing at least once in a 4-day period? This math is used to guarantee system reliability (nines of availability).

---

### **Statistics Lesson 3: Akuna Capital - "Consecutive Fives"**

#### **The Problem**
What is the expected number of times you must roll a fair six-sided die to get two consecutive 5s?

#### **The Logic (Markov Chains / State Transitions)**
Let E be the expected number of rolls. Break this down by states:
1. Roll a non-5: Probability is 5/6. Wasted 1 roll. Expected additional rolls: E + 1.
2. Roll a 5, then a non-5: Probability is (1/6) * (5/6) = 5/36. Wasted 2 rolls. Expected additional rolls: E + 2.
3. Roll a 5, then a 5: Probability is (1/6) * (1/6) = 1/36. Succeeded in 2 rolls. 

Equation: E = (5/6)*(E + 1) + (5/36)*(E + 2) + (1/36)*(2)
Multiply by 36: 36E = 30(E + 1) + 5(E + 2) + 2
36E = 35E + 42 -> E = 42.
**Answer: 42 rolls.**

#### **Senior Data Engineer Perspective**
* **Exponential Backoff and Retries:** This math reflects system retry logic. If hitting an external API randomly times out, how many total attempts should your Airflow DAG expect to make before hitting consecutive failures?

---

### **Statistics Lesson 4: D.E. Shaw - "Biased Coin?"**

#### **The Problem**
You have N coins. One is biased (heads on both sides). The other N-1 are fair. You draw a coin at random, flip it k times, and get k heads. What is the probability you picked the biased coin?

#### **The Logic (Bayes' Theorem)**
Let B = Biased coin (P(B) = 1/N). Let F = Fair coin (P(F) = (N-1)/N). Let Hk = Event of k consecutive heads.
Find P(B | Hk) = P(Hk | B) * P(B) / [ P(Hk | B) * P(B) + P(Hk | F) * P(F) ]
P(Hk | B) = 1. P(Hk | F) = (1/2)^k.
P(B | Hk) = (1 * 1/N) / [ (1 * 1/N) + ((1/2)^k * (N-1)/N) ]
Multiply by N: 1 / [ 1 + (N-1)/(2^k) ]

#### **Senior Data Engineer Perspective**
* **Alert Fatigue:** If your monitoring system sends an alert, what is the probability it is a real outage (Biased coin) vs a temporary CPU spike on a healthy database (Fair coin)? Bayes' Theorem is the antidote to false positive pager alerts.

---

### **Statistics Lesson 5: Facebook - "Product vs. Square"**

#### **The Problem**
Let X and Y be two independent standard normal random variables (Mean = 0, Variance = 1). Which is greater: the expected value of X^2, or the expected value of X*Y?

#### **The Logic (Expected Values of Random Variables)**
1. Because X is a standard normal distribution, its variance is E[X^2] - (E[X])^2.
2. Since E[X] = 0, the Variance is exactly equal to E[X^2]. Therefore, E[X^2] = 1.
3. Because X and Y are independent, the expected value of their product is the product of their expected values: E[X*Y] = E[X] * E[Y].
4. Since both have a mean of 0, E[X*Y] = 0 * 0 = 0.
**Answer: E[X^2] is greater (1 > 0).**

#### **Senior Data Engineer Perspective**
* **Feature Engineering Pipeline Costs:** If data scientists ask you to generate polynomial features (squaring variables) versus interaction features (multiplying variables), squaring variables fundamentally shifts the expected distribution and variance of the data much more drastically. This impacts scaling limits and data type precision (e.g., FLOAT32 vs FLOAT64 overflow limits) in your data warehouse.

---

### **Statistics Lesson 6: Google - "Minimum of Two Uniform Variables"**

#### **The Problem**
Let X and Y be two independent random variables, both uniformly distributed between 0 and 1. What is the expected value of the minimum of X and Y?

#### **The Logic (CDF and PDF Integrations)**
Let Z = min(X, Y). We want to find E[Z].
1. First, find the probability that Z is greater than some value z. For the minimum to be greater than z, BOTH X and Y must be greater than z.
2. P(Z > z) = P(X > z) * P(Y > z) = (1 - z) * (1 - z) = (1 - z)^2.
3. The Cumulative Density Function (CDF) is 1 - P(Z > z) = 1 - (1 - z)^2 = 2z - z^2.
4. The Probability Density Function (PDF) is the derivative: 2 - 2z.
5. The expected value is the integral of z * PDF from 0 to 1: Integral(2z - 2z^2) = z^2 - (2/3)z^3. Evaluated from 0 to 1, this equals 1 - 2/3 = 1/3.
**Answer: 1/3.**

#### **Senior Data Engineer Perspective**
* **Distributed System Latency:** If you launch a Spark job that requires two parallel executors to fetch data from an API, and their response times are uniformly distributed, this math tells you the expected time for the *first* executor to return. (Conversely, the *max* of X and Y is 2/3, which is the expected time for the *entire* job to finish waiting for the slowest node).

---

### **Statistics Lesson 7: Microsoft - "Frequentist vs. Bayesian"**

#### **The Problem**
Explain the fundamental difference between Frequentist and Bayesian statistics in simple terms.

#### **The Logic (Philosophical Foundations)**
* **Frequentist:** Probability is the long-run frequency of an event happening. The parameters of the universe are fixed, and the data we observe is random. (e.g., "If I flip this coin infinite times, it will be heads 50% of the time"). It does not use prior beliefs.
* **Bayesian:** Probability is a degree of belief or certainty. The data we observe is fixed (because it already happened), and the parameters of the universe are random variables. (e.g., "Based on my prior knowledge of coins, and the 5 flips I just saw, I am 95% certain this coin is fair"). It updates beliefs as new data arrives using Bayes' Theorem.

#### **Senior Data Engineer Perspective**
* **Streaming vs. Batch Analytics:** Frequentist statistics are often used in traditional Batch A/B testing platforms (calculating standard p-values at the end of a 2-week test). Bayesian statistics are the backbone of modern Real-Time / Streaming ML pipelines (like Multi-Armed Bandits), where the system continuously updates its "belief" about the best algorithm in real-time as a Kafka stream of user clicks flows in.